# SQL → построчный DataFrame → агрегат для дата-каталога

Ноутбук принимает `queries_df` с колонками `query_text` и `query_text_template`. Главные результаты:

- `row_analysis_df`: ровно одна строка на входную строку, разбор лежит во вложенном поле `analysis`;
- `aggregate_df`: одна строка на `schema + table + column + value + context + operator + pattern`;
- HTML не строится, пока явно не указано `build_html=True`.

SQL только разбирается как AST и **не выполняется**.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'gp_sql_analyzer').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gp_sql_analyzer.dataframe import analyze_dataframe

DEMO_OUTPUT = ROOT / 'artifacts' / 'notebook-demo'
ROOT, DEMO_OUTPUT

(PosixPath('/Users/family/Documents/sqlglot'),
 PosixPath('/Users/family/Documents/sqlglot/artifacts/notebook-demo'))

## 1. Входной DataFrame

Ниже небольшой демонстрационный набор на таблицах TPC-DS. Замените всю клетку своим DataFrame:

```python
queries_df = your_dataframe[['query_text', 'query_text_template']].copy()
```

`query_id` и `source_row_count` необязательны. `source_row_count` полезен, если одинаковые пары SQL заранее сгруппированы.

In [2]:
q44_style = '''
SELECT ss_item_sk
FROM store_sales AS ss1
WHERE ss_store_sk = 4
GROUP BY ss_item_sk
HAVING avg(ss_net_profit) > 0.9 * (
  SELECT avg(ss_net_profit)
  FROM store_sales
  WHERE ss_store_sk = 4 AND ss_addr_sk IS NULL
  GROUP BY ss_store_sk
)
'''

pattern_query = '''
WITH filtered AS (
  SELECT i_color, d_month_seq
  FROM item
  JOIN date_dim ON i_item_sk = d_date_sk
  WHERE i_color ILIKE '%purple%'
     OR i_color ~* '^(red|blue)$'
     OR d_month_seq BETWEEN 1200 AND 1200 + 11
)
SELECT * FROM filtered
'''
pattern_template = pattern_query.replace('%purple%', '%&CHARACTER%').replace(
    '^(red|blue)$', '&CHARACTER'
)

queries_df = pd.DataFrame([
    {
        'query_id': 'tpcds-q66-fragment',
        'query_text': "SELECT * FROM ship_mode WHERE sm_carrier IN ('DHL', 'BARIAN')",
        'query_text_template': "SELECT * FROM ship_mode WHERE sm_carrier IN ('&CHARACTER', '&CHARACTER')",
        'source_row_count': 2,
    },
    {
        'query_id': 'tpcds-q44-style',
        'query_text': q44_style,
        'query_text_template': q44_style,
        'source_row_count': 1,
    },
    {
        'query_id': 'tpcds-patterns',
        'query_text': pattern_query,
        'query_text_template': pattern_template,
        'source_row_count': 3,
    },
])

display(queries_df[['query_id', 'source_row_count']])

,query_id,source_row_count
0,tpcds-q66-fragment,2
1,tpcds-q44-style,1
2,tpcds-patterns,3


## 2. Необязательный снимок схемы

Для точного lineage передайте колонки из `information_schema.columns` или `pg_catalog`. Без `schema_df` анализ работает, но неквалифицированные колонки могут остаться `ambiguous/unresolved`.

In [3]:
schema_rows = {
    'ship_mode': ['sm_carrier'],
    'store_sales': ['ss_item_sk', 'ss_net_profit', 'ss_store_sk', 'ss_addr_sk'],
    'item': ['i_item_sk', 'i_color'],
    'date_dim': ['d_date_sk', 'd_month_seq'],
}
schema_df = pd.DataFrame([
    {
        'table_catalog': 'warehouse',
        'table_schema': 'tpcds',
        'table_name': table_name,
        'column_name': column_name,
    }
    for table_name, columns in schema_rows.items()
    for column_name in columns
])
display(schema_df.head())

,table_catalog,table_schema,table_name,column_name
0,warehouse,tpcds,ship_mode,sm_carrier
1,warehouse,tpcds,store_sales,ss_item_sk
2,warehouse,tpcds,store_sales,ss_net_profit
3,warehouse,tpcds,store_sales,ss_store_sk
4,warehouse,tpcds,store_sales,ss_addr_sk


## 3. Анализ

Основной вызов возвращает DataFrame-объекты. Указан `output_dir`, поэтому дополнительно сохраняются JSON/JSONL. HTML остаётся выключен.

In [4]:
result = analyze_dataframe(
    queries_df,
    schema_df=schema_df,
    default_schema='tpcds',
    output_dir=DEMO_OUTPUT,
    build_html=False,
    example_limit=20,
)

row_analysis_df = result.row_analysis_df
aggregate_df = result.aggregate_df

assert len(row_analysis_df) == len(queries_df)
assert 'html' not in result.artifact_paths
print(f'Строк на входе и выходе: {len(row_analysis_df)}')
print(f'Строк в aggregate_df: {len(aggregate_df)}')

Строк на входе и выходе: 3
Строк в aggregate_df: 9


## 4. Построчный разбор

Колонка `analysis` содержит список всех событий конкретного SQL. Здесь же видны статусы качества.

In [5]:
display(row_analysis_df[[
    'query_id', 'analysis_status', 'analysis_count',
    'resolved_count', 'multi_source_count', 'ambiguous_count',
    'unresolved_count', 'analysis',
]])

,query_id,analysis_status,analysis_count,resolved_count,multi_source_count,ambiguous_count,unresolved_count,analysis
0,tpcds-q66-fragment,ok,2,2,0,0,0,"[{'query_id': 'tpcds-q66-fragment', 'query_has..."
1,tpcds-q44-style,ok,4,4,0,0,0,"[{'query_id': 'tpcds-q44-style', 'query_hash':..."
2,tpcds-patterns,ok,4,4,0,0,0,"[{'query_id': 'tpcds-patterns', 'query_hash': ..."


In [6]:
q44_events = row_analysis_df.loc[
    row_analysis_df['query_id'].eq('tpcds-q44-style'), 'analysis'
].iloc[0]
pd.DataFrame(q44_events)[[
    'base_columns', 'lineage_status', 'clause_context',
    'operator_or_function', 'extracted_value', 'origin',
]]

,base_columns,lineage_status,clause_context,operator_or_function,extracted_value,origin
0,[warehouse.tpcds.store_sales.ss_addr_sk],resolved,WHERE,IS NULL,NULL,null_check
1,[warehouse.tpcds.store_sales.ss_store_sk],resolved,WHERE,=,4,original_literal
2,[warehouse.tpcds.store_sales.ss_net_profit],resolved,HAVING,>,0.9,original_literal
3,[warehouse.tpcds.store_sales.ss_store_sk],resolved,WHERE,=,4,original_literal


## 5. Агрегирующий DataFrame

В него входят только однозначно разрешённые физические колонки. `source_row_count` учитывает вес исходных строк.

In [7]:
display(aggregate_df[[
    'qualified_name', 'extracted_value', 'clause_context',
    'operator_or_function', 'pattern_family',
    'source_row_count', 'distinct_query_count', 'example_query_ids',
]])

,qualified_name,extracted_value,clause_context,operator_or_function,pattern_family,source_row_count,distinct_query_count,example_query_ids
0,warehouse.tpcds.date_dim.d_month_seq,1200,WHERE,BETWEEN,exact_value,3,1,[tpcds-patterns]
1,warehouse.tpcds.date_dim.d_month_seq,1200 + 11,WHERE,BETWEEN,exact_value,3,1,[tpcds-patterns]
2,warehouse.tpcds.item.i_color,^(red|blue)$,WHERE,~*,regex,3,1,[tpcds-patterns]
3,warehouse.tpcds.item.i_color,purple,WHERE,ILIKE,like_contains,3,1,[tpcds-patterns]
4,warehouse.tpcds.ship_mode.sm_carrier,BARIAN,WHERE,IN,exact_value,2,1,[tpcds-q66-fragment]
5,warehouse.tpcds.ship_mode.sm_carrier,DHL,WHERE,IN,exact_value,2,1,[tpcds-q66-fragment]
6,warehouse.tpcds.store_sales.ss_addr_sk,NULL,WHERE,IS NULL,null_check,1,1,[tpcds-q44-style]
7,warehouse.tpcds.store_sales.ss_net_profit,0.9,HAVING,>,exact_value,1,1,[tpcds-q44-style]
8,warehouse.tpcds.store_sales.ss_store_sk,4,WHERE,=,exact_value,2,1,[tpcds-q44-style]


In [8]:
patterns_df = aggregate_df.loc[
    aggregate_df['pattern_family'].ne('exact_value'),
    ['qualified_name', 'extracted_value', 'pattern_family', 'pattern_format', 'source_row_count'],
]
display(patterns_df)

,qualified_name,extracted_value,pattern_family,pattern_format,source_row_count
2,warehouse.tpcds.item.i_color,^(red|blue)$,regex,alternation+anchored_end+anchored_start+case_i...,3
3,warehouse.tpcds.item.i_color,purple,like_contains,like_contains,3
6,warehouse.tpcds.store_sales.ss_addr_sk,NULL,null_check,null_check,1


## 6. Табличный срез дата-каталога и ошибки

In [9]:
display(result.catalog_tables_df[[
    'qualified_name', 'column_count', 'active_column_count',
    'source_row_count', 'distinct_query_count',
]])
display(result.catalog_columns_df[[
    'qualified_name', 'usage_status', 'source_row_count',
    'context_counts', 'operator_counts', 'top_values',
]])
display(result.errors_df)

,qualified_name,column_count,active_column_count,source_row_count,distinct_query_count
0,warehouse.tpcds.date_dim,2,1,6,1
1,warehouse.tpcds.item,2,1,6,1
2,warehouse.tpcds.ship_mode,1,1,4,1
3,warehouse.tpcds.store_sales,4,3,4,1


,qualified_name,usage_status,source_row_count,context_counts,operator_counts,top_values
0,warehouse.tpcds.date_dim.d_date_sk,unused,0,{},{},[]
1,warehouse.tpcds.date_dim.d_month_seq,active,6,{'WHERE': 6},{'BETWEEN': 6},"[{'value': '1200', 'pattern_family': 'exact_va..."
2,warehouse.tpcds.item.i_color,active,6,{'WHERE': 6},"{'ILIKE': 3, '~*': 3}","[{'value': '^(red|blue)$', 'pattern_family': '..."
3,warehouse.tpcds.item.i_item_sk,unused,0,{},{},[]
4,warehouse.tpcds.ship_mode.sm_carrier,active,4,{'WHERE': 4},{'IN': 4},"[{'value': 'BARIAN', 'pattern_family': 'exact_..."
5,warehouse.tpcds.store_sales.ss_addr_sk,active,1,{'WHERE': 1},{'IS NULL': 1},"[{'value': 'NULL', 'pattern_family': 'null_che..."
6,warehouse.tpcds.store_sales.ss_item_sk,unused,0,{},{},[]
7,warehouse.tpcds.store_sales.ss_net_profit,active,1,{'HAVING': 1},{'>': 1},"[{'value': '0.9', 'pattern_family': 'exact_val..."
8,warehouse.tpcds.store_sales.ss_store_sk,active,2,{'WHERE': 2},{'=': 2},"[{'value': '4', 'pattern_family': 'exact_value..."


,query_id,stage,error_type,message,sql_fragment


In [10]:
print('Созданные файлы:')
for name, path in result.artifact_paths.items():
    print(f'  {name:16s} {path}')
assert not (DEMO_OUTPUT / 'catalog-stats.html').exists()

Созданные файлы:
  row_analysis     /Users/family/Documents/sqlglot/artifacts/notebook-demo/row_analysis.jsonl
  details          /Users/family/Documents/sqlglot/artifacts/notebook-demo/details.jsonl
  errors           /Users/family/Documents/sqlglot/artifacts/notebook-demo/errors.jsonl
  aggregate        /Users/family/Documents/sqlglot/artifacts/notebook-demo/aggregate.jsonl
  catalog_json     /Users/family/Documents/sqlglot/artifacts/notebook-demo/catalog-stats.json
  catalog_columns  /Users/family/Documents/sqlglot/artifacts/notebook-demo/catalog-columns.jsonl
  schema           /Users/family/Documents/sqlglot/artifacts/notebook-demo/schema.json


## 7. Необязательно: загрузка из Greenplum

Следующая клетка намеренно выключена. Подключение берёт `GP_HOST`, `GP_PORT`, `GP_DBNAME`, `GP_USER`, `GP_PASSWORD`, `GP_SSLMODE` из окружения. Запросы только читают данные.

In [11]:
if False:  # поменяйте на True только в своей среде Greenplum
    from gp_sql_analyzer.greenplum import connect_greenplum

    connection = connect_greenplum()  # read-only session
    queries_df = pd.read_sql_query(
        '''
        SELECT query_text, query_text_template, count(*)::bigint AS source_row_count
        FROM analytics.query_log
        WHERE query_text IS NOT NULL AND query_text_template IS NOT NULL
        GROUP BY query_text, query_text_template
        ''',
        connection,
    )
    schema_df = pd.read_sql_query(
        '''
        SELECT current_database() AS table_catalog, n.nspname AS table_schema,
               c.relname AS table_name, a.attname AS column_name
        FROM pg_catalog.pg_class c
        JOIN pg_catalog.pg_namespace n ON n.oid = c.relnamespace
        JOIN pg_catalog.pg_attribute a ON a.attrelid = c.oid
        WHERE c.relkind IN ('r', 'v', 'm', 'f')
          AND a.attnum > 0 AND NOT a.attisdropped
        ''',
        connection,
    )

## 8. Необязательно: HTML

HTML создаётся только отдельным явным вызовом:

```python
result_with_html = analyze_dataframe(
    queries_df, schema_df=schema_df, default_schema='tpcds',
    output_dir=DEMO_OUTPUT / 'with-html', build_html=True,
)
result_with_html.artifact_paths['html']
```